# 🧠 Heady — 3D Vector Space GPU Runtime

**Colab Pillar** of the Heady Unified Platform.

This notebook provides GPU-accelerated embedding generation and 3D vector operations.
It connects to the Cloud Run MCP bridge for real-time vector processing.

---

## Architecture
- **Embedding Model:** sentence-transformers/all-MiniLM-L6-v2 (384D)
- **3D Projection:** PCA-lite → (x, y, z) → 8 octant zones
- **GPU:** CUDA-accelerated batch embedding
- **API:** REST endpoint for Cloud Run bridge


In [ ]:
# ── Install Dependencies ──────────────────────────────────
!pip install -q sentence-transformers flask pyngrok torch numpy

In [ ]:
# ── GPU Check ────────────────────────────────────────────
import torch
print(f'🖥️ GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   Device: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('   ⚠️ Running on CPU — embeddings will be slower')

In [ ]:
# ── Load Embedding Model ─────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'📥 Loading {MODEL_NAME} on {device}...')
start = time.time()
model = SentenceTransformer(MODEL_NAME, device=device)
print(f'✅ Model loaded in {time.time() - start:.1f}s')
print(f'   Dimensions: {model.get_sentence_embedding_dimension()}')

In [ ]:
# ── 3D Projection Functions ───────────────────────────────

def to_3d(embedding):
    """PCA-lite: 384D → (x, y, z) via group averaging."""
    third = len(embedding) // 3
    x = np.mean(embedding[:third])
    y = np.mean(embedding[third:2*third])
    z = np.mean(embedding[2*third:])
    return {'x': float(x), 'y': float(y), 'z': float(z)}

def assign_zone(x, y, z):
    """Map 3D coords to octant zone (0-7)."""
    return (1 if x >= 0 else 0) | (2 if y >= 0 else 0) | (4 if z >= 0 else 0)

def embed_batch(texts):
    """Generate embeddings with 3D projection."""
    embeddings = model.encode(texts, show_progress_bar=False)
    results = []
    for i, emb in enumerate(embeddings):
        pos = to_3d(emb)
        zone = assign_zone(pos['x'], pos['y'], pos['z'])
        results.append({
            'embedding': emb.tolist(),
            '_3d': pos,
            '_zone': zone,
            'text': texts[i][:100]
        })
    return results

# Quick test
test = embed_batch(['Hello from Heady 3D vector space'])
print(f'✅ Embedding test: zone={test[0]["_zone"]}, 3D={test[0]["_3d"]}')

In [ ]:
# ── GPU Embedding Server ─────────────────────────────────
# Provides REST API for the Cloud Run MCP bridge

from flask import Flask, request, jsonify
from threading import Thread
import json

app = Flask(__name__)
stats = {'requests': 0, 'embeddings': 0, 'errors': 0}

@app.route('/embed', methods=['POST'])
def embed_endpoint():
    try:
        data = request.json
        texts = data.get('texts', [])
        if isinstance(texts, str):
            texts = [texts]
        if not texts:
            return jsonify({'ok': False, 'error': 'No texts provided'}), 400
        
        start = time.time()
        embeddings = model.encode(texts, show_progress_bar=False)
        duration = time.time() - start
        
        stats['requests'] += 1
        stats['embeddings'] += len(texts)
        
        return jsonify({
            'ok': True,
            'embeddings': embeddings.tolist(),
            'count': len(texts),
            'dims': embeddings.shape[1],
            'duration_ms': round(duration * 1000),
            'device': device
        })
    except Exception as e:
        stats['errors'] += 1
        return jsonify({'ok': False, 'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'ok': True,
        'service': 'heady-colab-gpu',
        'model': MODEL_NAME,
        'device': device,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'stats': stats
    })

@app.route('/project', methods=['POST'])
def project_3d():
    """Project texts into 3D space with zone assignment."""
    try:
        data = request.json
        texts = data.get('texts', [])
        if isinstance(texts, str):
            texts = [texts]
        results = embed_batch(texts)
        return jsonify({'ok': True, 'results': results})
    except Exception as e:
        return jsonify({'ok': False, 'error': str(e)}), 500

print('🚀 Starting Heady GPU Embedding Server on port 5000...')
Thread(target=lambda: app.run(host='0.0.0.0', port=5000)).start()
print('✅ Server running at http://localhost:5000')

In [ ]:
# ── Expose via ngrok (connects to Cloud Run) ──────────────
# Set NGROK_AUTH_TOKEN in Colab secrets or paste below

import os

NGROK_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(5000)
    EMBEDDING_URL = tunnel.public_url
    print(f'\n🌐 Public URL: {EMBEDDING_URL}')
    print(f'\n📋 Set this in your Cloud Run environment:')
    print(f'   HEADY_EMBEDDING_URL={EMBEDDING_URL}')
    print(f'\n   gcloud run services update heady-manager \\')
    print(f'     --set-env-vars HEADY_EMBEDDING_URL={EMBEDDING_URL} \\')
    print(f'     --region us-central1')
else:
    print('⚠️ No NGROK_AUTH_TOKEN set.')
    print('   Add to Colab Secrets or set manually:')
    print('   os.environ["NGROK_AUTH_TOKEN"] = "your_token"')
    print('\n   For now, embedding server runs at http://localhost:5000')
    EMBEDDING_URL = 'http://localhost:5000'

In [ ]:
# ── 3D Vector Space Visualization ─────────────────────────
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Generate sample embeddings for visualization
sample_texts = [
    'Heady is a unified AI platform',
    'Sacred geometry orchestration engine',
    '3D vector memory with 8 octant zones',
    'Cross-device synchronization via WebSocket',
    'MCP bridge with 43 specialized tools',
    'HeadyBee template-driven task automation',
    'Google Cloud Run serverless compute',
    'Cloudflare edge workers for low latency',
    'Continuous learning from every interaction',
    'Vector projection tracking and telemetry',
    'GitHub monorepo source of truth',
    'GPU-accelerated embedding generation',
    'Hybrid RAG with graph relationships',
    'Memory importance scoring algorithm',
    'Perplexity Sonar deep research integration',
]

results = embed_batch(sample_texts)

# Plot
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

colors = ['#4c8fff', '#00d4ff', '#00ff88', '#88ff00',
          '#ff8800', '#ff0055', '#aa00ff', '#ff00ff']

for r in results:
    zone = r['_zone']
    ax.scatter(r['_3d']['x'], r['_3d']['y'], r['_3d']['z'],
              c=colors[zone], s=120, alpha=0.8, edgecolors='white', linewidth=0.5)
    ax.text(r['_3d']['x'], r['_3d']['y'], r['_3d']['z'],
            f'  {r["text"][:25]}...', fontsize=6, alpha=0.7)

ax.set_xlabel('X (dims 0-127)')
ax.set_ylabel('Y (dims 128-255)')
ax.set_zlabel('Z (dims 256-383)')
ax.set_title('Heady 3D Vector Space — Live GPU Projection', fontsize=14, fontweight='bold')
ax.set_facecolor('#0a0a1a')
fig.patch.set_facecolor('#0a0a1a')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.zaxis.label.set_color('white')
ax.title.set_color('white')

plt.tight_layout()
plt.savefig('heady_3d_vectors.png', dpi=150, facecolor='#0a0a1a')
plt.show()
print(f'\n✅ Plotted {len(results)} vectors across zones: {set(r["_zone"] for r in results)}')

In [ ]:
# ── Keep Alive ───────────────────────────────────────────
# Run this cell to keep the embedding server active

import time

print('🧠 Heady Colab GPU Runtime — Active')
print(f'   Embedding URL: {EMBEDDING_URL}')
print(f'   Model: {MODEL_NAME}')
print(f'   Device: {device}')
print('\n   Press Ctrl+C or stop cell to shut down\n')

try:
    while True:
        time.sleep(60)
        print(f'   ♥ Heartbeat | Requests: {stats["requests"]} | Embeddings: {stats["embeddings"]} | Errors: {stats["errors"]}')
except KeyboardInterrupt:
    print('\n🛑 Heady GPU Runtime stopped.')